# Vectorless RAG System using PageIndex - Student Improved Version

**Made by:** Darshan Nitin Barhate

This notebook shows a simple vectorless RAG pipeline. Instead of storing document chunks in a vector database, the project uses PageIndex to build a tree-like document structure. Then an LLM chooses the best document sections and answers using only those sections.

My improvements focus on making the project easier to run, easier to understand, and safer for a student GitHub submission.


## What I Improved

- Added clearer notebook sections so the flow is easier to follow.
- Removed hardcoded API keys and used `.env` variables instead.
- Added checks for missing keys and missing PDF files.
- Added helper functions for printing the document tree, counting nodes, compressing the tree, and parsing JSON safely.
- Renamed confusing financial-example rules into course/document routing rules.
- Added a simple evaluation table so answers can be compared across test questions.
- Cleared old notebook outputs so the file is cleaner for GitHub.


## 1. Install Dependencies

Run this cell once in a fresh environment. If the packages are already installed, you can skip it.


In [ ]:
# %pip install -U pageindex openai python-dotenv


## 2. Load API Keys and Settings

Create a `.env` file in the same folder as this notebook. Use `.env.example` as a guide.

Required values:

```text
PAGEINDEX_API_KEY=your_pageindex_key_here
OPENAI_API_KEY=your_openai_key_here
PDF_PATH=./sample_document.pdf
```


In [ ]:
import json
import os
import time
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PDF_PATH = Path(os.getenv("PDF_PATH", "./sample_document.pdf"))
DEFAULT_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o")


def require_setting(name: str, value: str | None) -> str:
    """Stop early with a helpful message when an important setting is missing."""
    if not value:
        raise ValueError(f"Missing {name}. Add it to your .env file before running the notebook.")
    return value

PAGEINDEX_API_KEY = require_setting("PAGEINDEX_API_KEY", PAGEINDEX_API_KEY)
OPENAI_API_KEY = require_setting("OPENAI_API_KEY", OPENAI_API_KEY)

print("Settings loaded")
print(f"PDF path: {PDF_PATH}")
print(f"Model: {DEFAULT_MODEL}")


## 3. Create Clients

These clients connect the notebook to PageIndex and OpenAI.


In [ ]:
from pageindex import PageIndexClient
from openai import OpenAI

pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)
openai_client = OpenAI(api_key=OPENAI_API_KEY)

print("PageIndex client ready")
print("OpenAI client ready")


## 4. Upload the PDF and Build the PageIndex Tree

The PDF is uploaded to PageIndex. PageIndex turns the document into a tree of sections, similar to a smart table of contents.


In [ ]:
if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {PDF_PATH}. Put your PDF in this folder or update PDF_PATH in .env."
    )

print(f"Uploading: {PDF_PATH}")
result = pi_client.submit_document(str(PDF_PATH))
doc_id = result["doc_id"]
print(f"Uploaded document. Document ID: {doc_id}")


In [ ]:
def wait_for_pageindex_tree(doc_id: str, sleep_seconds: int = 5, max_checks: int = 60) -> None:
    """Wait until PageIndex finishes processing the uploaded document."""
    print("Building tree index...")
    for attempt in range(1, max_checks + 1):
        status_result = pi_client.get_document(doc_id)
        status = status_result.get("status")
        print(f"Check {attempt:02d}: {status}")
        if status == "completed":
            print("Tree index ready")
            return
        if status == "failed":
            raise RuntimeError("PageIndex processing failed. Try another PDF or check the PDF format.")
        time.sleep(sleep_seconds)
    raise TimeoutError("PageIndex did not finish in time. Try again later or increase max_checks.")

wait_for_pageindex_tree(doc_id)


In [ ]:
tree_result = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"Top-level sections: {len(pageindex_tree)}")
if pageindex_tree:
    print(json.dumps(pageindex_tree[0], indent=2)[:1000])


## 5. Explore the Document Tree

These helper functions make the tree easier to read.


In [ ]:
def print_tree(nodes: list, indent: int = 0) -> None:
    for node in nodes:
        prefix = "  " * indent + ("- " if indent > 0 else "")
        page = node.get("page_index", "?")
        title = node.get("title", "Untitled")
        node_id = node.get("node_id", "no-id")
        print(f"{prefix}[{node_id}] {title} (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)


def count_nodes(nodes: list) -> int:
    total = len(nodes)
    for node in nodes:
        total += count_nodes(node.get("nodes", []))
    return total

print("Full document structure:\n")
print_tree(pageindex_tree)
print(f"\nTotal nodes in tree: {count_nodes(pageindex_tree)}")


## 6. Vectorless Retrieval

The LLM reads the tree and chooses the most useful node IDs. This replaces the normal vector search step.


In [ ]:
def compress_tree(nodes: list, summary_chars: int = 180) -> list:
    """Keep only the details the LLM needs for selecting useful sections."""
    compressed = []
    for node in nodes:
        item = {
            "node_id": node.get("node_id"),
            "title": node.get("title", "Untitled"),
            "page": node.get("page_index", "?"),
            "summary": node.get("text", "")[:summary_chars],
        }
        if node.get("nodes"):
            item["children"] = compress_tree(node["nodes"], summary_chars)
        compressed.append(item)
    return compressed


def safe_json_loads(text: str) -> dict:
    """Parse model JSON and return a useful fallback if parsing fails."""
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"thinking": "Model did not return valid JSON.", "node_list": []}


In [ ]:
def llm_tree_search(query: str, tree: list, model: str = DEFAULT_MODEL) -> dict:
    compressed_tree = compress_tree(tree)
    prompt = f"""You are given a question and a document tree.
Choose the node IDs that most likely contain the answer.

Question: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Return only JSON in this format:
{{
  "thinking": "short reason for the selected nodes",
  "node_list": ["node_id1", "node_id2"]
}}"""
    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
    )
    return safe_json_loads(response.choices[0].message.content)


def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    found = []
    for node in tree:
        if node.get("node_id") in target_ids:
            found.append(node)
        found.extend(find_nodes_by_ids(node.get("nodes", []), target_ids))
    return found


## 7. Generate the Final Answer

The answer is created only from the retrieved sections. This reduces guessing because the model is told to use the selected context.


In [ ]:
def generate_answer(query: str, nodes: list, model: str = DEFAULT_MODEL) -> str:
    if not nodes:
        return "No relevant sections were found in the document."

    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: {node.get('title', 'Untitled')} | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""Answer the question using only the context below.
Cite the section title and page number when possible.
Keep the answer clear and simple.

Question: {query}

Context:
{context}

Answer:"""
    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content


def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    if verbose:
        print("=" * 60)
        print(f"Query: {query}")
        print("=" * 60)

    search_result = llm_tree_search(query, tree)
    node_ids = search_result.get("node_list", [])
    nodes = find_nodes_by_ids(tree, node_ids)

    if verbose:
        print(f"Reasoning: {search_result.get('thinking', 'N/A')}")
        print(f"Retrieved node IDs: {node_ids}")
        print(f"Sections found: {[node.get('title', 'Untitled') for node in nodes]}")

    answer = generate_answer(query, nodes)
    if verbose:
        print("\nAnswer:\n")
        print(answer)
    return answer


## 8. Try a Question

Change the question below to match your own PDF.


In [ ]:
answer = vectorless_rag(
    query="What are the main topics covered in this document?",
    tree=pageindex_tree,
)


## 9. Add Simple Expert Routing Rules

These rules help the LLM choose better sections when the document has known topics. I changed the old finance example into a general course/document example so it matches this project better.


In [ ]:
COURSE_EXPERT_RULES = """
Route questions to the best matching document sections.

General examples:
- overview, introduction, purpose -> introduction or summary sections
- syllabus, modules, weekly plan -> course outline or module sections
- fine-tuning, LoRA, QLoRA, RLHF -> modern LLM fine-tuning sections
- RAG, retrieval, chunking, reranking -> RAG or retrieval sections
- tokenization, BPE, WordPiece -> tokenization sections
- evaluation, benchmarks, testing -> evaluation sections
- deployment, serving, quantization -> deployment or production sections
"""


def llm_tree_search_with_expert(
    query: str,
    tree: list,
    expert_rules: str,
    model: str = DEFAULT_MODEL,
) -> dict:
    prompt = f"""You are a document routing assistant.
Use the expert rules and the document tree to choose the best node IDs.

Question: {query}

Expert Rules:
{expert_rules}

Document Tree:
{json.dumps(compress_tree(tree), indent=2)}

Return only JSON in this format:
{{
  "thinking": "short reason using the expert rules",
  "node_list": ["node_id1", "node_id2"]
}}"""
    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
    )
    return safe_json_loads(response.choices[0].message.content)


def expert_rag(query: str, tree: list, rules: str = COURSE_EXPERT_RULES) -> str:
    result = llm_tree_search_with_expert(query, tree, rules)
    nodes = find_nodes_by_ids(tree, result.get("node_list", []))
    return generate_answer(query, nodes)


In [ ]:
question = "Explain the most important ideas in the document."
print(expert_rag(question, pageindex_tree))


## 10. Simple Evaluation Table

This is not a full research evaluation, but it helps a student show that the project was tested with more than one question.


In [ ]:
def run_test_questions(questions: list[str], tree: list) -> list[dict]:
    rows = []
    for question in questions:
        search_result = llm_tree_search(question, tree)
        node_ids = search_result.get("node_list", [])
        nodes = find_nodes_by_ids(tree, node_ids)
        answer = generate_answer(question, nodes)
        rows.append({
            "question": question,
            "selected_nodes": node_ids,
            "sections": [node.get("title", "Untitled") for node in nodes],
            "answer_preview": answer[:250],
        })
    return rows


test_questions = [
    "What is this document mainly about?",
    "What are the key topics or modules?",
    "What should a beginner learn first?",
]

evaluation_rows = run_test_questions(test_questions, pageindex_tree)
for row in evaluation_rows:
    print(json.dumps(row, indent=2))
    print("-" * 60)


## 11. Optional: Use PageIndex Chat API Directly

This section is optional. It lets you compare the custom vectorless RAG pipeline with the PageIndex chat API.


In [ ]:
question = "What are the key findings in this document?"
response = pi_client.chat_completions(
    messages=[{"role": "user", "content": question}],
    doc_id=doc_id,
)
chat_answer = response["choices"][0]["message"]["content"]
print(chat_answer)


## 12. Final Notes

The main idea of this project is simple:

1. Upload a document.
2. Let PageIndex build a tree of the document.
3. Ask the LLM to choose useful tree nodes.
4. Generate an answer using only those selected sections.

This avoids a vector database and makes retrieval easier to inspect because we can see exactly which sections were selected.
